# 03 - DataFrames, SQL, UDF et export Parquet (Partie 3, Personne 1)

In [1]:
import sys
sys.path.append('/home/jovyan/work/pipeline')
from pedali_spark import get_spark
import df_analysis as da

spark = get_spark('pedali-03-dataframes')
stations, trips = da.load(spark)
res = da.analyses(spark, stations, trips)
trips.printSchema()
print('Trajets valides :', trips.count())

root
 |-- start_station_id: string (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- bike_id: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- duration_min: integer (nullable = true)
 |-- price_mad: double (nullable = true)
 |-- status: string (nullable = true)



Trajets valides : 19709


## Agrégations DataFrame

In [2]:
for name in ['revenus_par_ville','top5_stations','repartition_duree_udf','statuts']:
    print('--', name)
    res[name].show(truncate=False)

-- revenus_par_ville


+----------+----------------+-------+-----------------+
|city      |chiffre_affaires|trajets|duree_moyenne_min|
+----------+----------------+-------+-----------------+
|Casablanca|157812.0        |5969   |17.63            |
|Marrakech |144175.5        |5484   |17.53            |
|Rabat     |163402.5        |6220   |17.51            |
+----------+----------------+-------+-----------------+

-- top5_stations


+----------------+------------+-------+
|start_station_id|station     |departs|
+----------------+------------+-------+
|S01             |Casa Port   |2815   |
|S11             |Rabat Ville |2142   |
|S10             |Hay Riad    |1938   |
|S05             |Jemaa el-Fna|1512   |
|S07             |Menara      |1485   |
+----------------+------------+-------+

-- repartition_duree_udf


+---------------+-------+
|categorie_duree|trajets|
+---------------+-------+
|courte (<10)   |2793   |
|longue (>=30)  |1129   |
|moyenne (10-29)|13751  |
+---------------+-------+

-- statuts


+---------+-------+----------------+
|status   |trajets|chiffre_affaires|
+---------+-------+----------------+
|breakdown|780    |0.0             |
|cancelled|1256   |0.0             |
|completed|17673  |465390.0        |
+---------+-------+----------------+



## SQL pur (mêmes résultats) et UDF enregistrée

In [3]:
for name in ['sql_revenus_par_ville','sql_top5_stations','sql_udf']:
    print('--', name)
    res[name].show(truncate=False)

-- sql_revenus_par_ville


+----------+----------------+-------+-----------------+
|city      |chiffre_affaires|trajets|duree_moyenne_min|
+----------+----------------+-------+-----------------+
|Casablanca|157812.0        |5969   |17.63            |
|Marrakech |144175.5        |5484   |17.53            |
|Rabat     |163402.5        |6220   |17.51            |
+----------+----------------+-------+-----------------+

-- sql_top5_stations


+----------------+------------+-------+
|start_station_id|station     |departs|
+----------------+------------+-------+
|S01             |Casa Port   |2815   |
|S11             |Rabat Ville |2142   |
|S10             |Hay Riad    |1938   |
|S05             |Jemaa el-Fna|1512   |
|S07             |Menara      |1485   |
+----------------+------------+-------+

-- sql_udf


+---------------+-------+
|categorie_duree|trajets|
+---------------+-------+
|courte (<10)   |2793   |
|longue (>=30)  |1129   |
|moyenne (10-29)|13751  |
+---------------+-------+



In [4]:
# Cohérence DataFrame <-> SQL
assert res['revenus_par_ville'].collect() == res['sql_revenus_par_ville'].collect()
assert res['top5_stations'].collect() == res['sql_top5_stations'].collect()
print('DataFrame == SQL : OK')

DataFrame == SQL : OK


## Export Parquet vers MinIO (`s3a://lake/export/parquet/`) et relecture

In [5]:
print(da.export_parquet(trips, res))
back = spark.read.parquet(da.EXPORT + '/trips_clean')
print('Relecture trips_clean :', back.count(), 'lignes ; partitions :', sorted(r[0] for r in back.select('city').distinct().collect()))

['trips_clean (partitionne par city)', 'revenus_par_ville', 'top5_stations', 'repartition_duree_udf', 'statuts', 'trajets_par_heure']


Relecture trips_clean : 19709 lignes ; partitions : ['Casablanca', 'Marrakech', 'Rabat']


In [6]:
spark.stop()